# 01 EDA: first look at the Backblaze Drive Stats sample

Data: `data/bronze_parquet` (2026 Q1 and Q2). All queries go through DuckDB; pandas only ever sees small results.
After each chart, write one sentence under "What I learned". Those sentences become feature ideas.

In [1]:
from pathlib import Path
import duckdb
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "data").exists():
    ROOT = ROOT.parent
GLOB = (ROOT / "data" / "bronze_parquet" / "*" / "part.parquet").as_posix()
SRC = f"read_parquet('{GLOB}', union_by_name=true, hive_partitioning=false)"
con = duckdb.connect()

# Change these to widen or narrow the development sample
DEV_MODELS = ["TOSHIBA MG07ACA14TA", "ST12000NM0008", "TOSHIBA MG08ACA16TA"]
print("repo root:", ROOT)

repo root: d:\Repositeries\Backblaze\drivewatch\drivewatch


## Q1. Fleet size per day
Does the fleet grow, shrink, or have gaps?

In [2]:
fleet = con.sql(f"""
    SELECT CAST(date AS DATE) AS day, count(*) AS drives
    FROM {SRC} GROUP BY 1 ORDER BY 1
""").df()
px.line(fleet, x="day", y="drives", title="Drives reporting per day").show()
print(fleet["drives"].describe())

count       181.000000
mean     345665.674033
std        6398.436546
min      336407.000000
25%      339021.000000
50%      345596.000000
75%      351467.000000
max      355284.000000
Name: drives, dtype: float64


What I learned: The fleet grew about 5% over the half-year, from roughly 339k to 355k drives per day, with a small dip in mid-January (to about 336k). No days are missing, since 181 days are present. So drives are constantly being added and retired, which means drive age and “first seen” matter as features, and I should expect new models to appear over time (drift).

## Q2. Failure rate by model
Annualised failure rate (AFR) = failures / drive-days x 365 x 100. Only models with at least 1,000 drives are shown, because small models give noisy rates.

In [3]:
afr = con.sql(f"""
    SELECT model,
           count(DISTINCT serial_number) AS drives,
           count(*) AS drive_days,
           sum(try_cast(failure AS INTEGER)) AS failures,
           round(sum(try_cast(failure AS INTEGER)) * 365.0 * 100 / count(*), 2) AS afr_pct
    FROM {SRC}
    GROUP BY model
    HAVING count(DISTINCT serial_number) >= 1000
    ORDER BY afr_pct DESC
""").df()
display(afr)
px.bar(afr, x="model", y="afr_pct", hover_data=["drives", "failures"],
       title="Annualised failure rate by model (%)").show()

,model,drives,drive_days,failures,afr_pct
0,ST14000NM0138,1251,222119,40.0,6.57
1,HGST HUH721212ALN604,9790,1731549,275.0,5.80
2,WDC WUH721816ALE6L0,2994,536743,56.0,3.81
3,ST24000NM002H,9739,1681677,156.0,3.39
4,TOSHIBA MG08ACA16TEY,4885,874306,72.0,3.01
5,HGST HUH721212ALE604,13283,2373956,196.0,3.01
6,ST12000NM0008,18652,3335615,274.0,3.00
7,ST8000NM0055,13241,2354160,151.0,2.34
8,ST8000DM002,8912,1202683,63.0,1.91
9,TOSHIBA MG07ACA14TA,37265,6712648,273.0,1.48


What I learned: Failure rates vary a lot by model, from about 0.4% a year to 5.8% (for models with a meaningful number of failures), against a fleet average of roughly 1.5%. So drive model must be a feature, and I should check that my dev sample mixes high-, mid- and low-failure models.

## Q3. Class balance
How rare are positives? Rough count of drive-days that have a failure 1 to 30 days ahead. The formal label is built in Phase 3; this is only a size estimate. Rows in the last 30 days are excluded from the denominator, because their 30-day window is not fully observed.

In [4]:
balance = con.sql(f"""
    WITH f AS (
        SELECT serial_number, CAST(date AS DATE) AS fdate FROM {SRC} WHERE failure = '1'
    ),
    d AS (
        SELECT serial_number, CAST(date AS DATE) AS d FROM {SRC}
    ),
    mx AS (SELECT max(d) AS last_day FROM d),
    pos AS (
        SELECT count(*) AS positives
        FROM d JOIN f USING (serial_number)
        WHERE date_diff('day', d.d, f.fdate) BETWEEN 1 AND 30
    ),
    tot AS (
        SELECT count(*) AS rows_labelable FROM d, mx WHERE d.d <= mx.last_day - 30
    )
    SELECT positives, rows_labelable, round(100.0 * positives / rows_labelable, 4) AS positive_pct
    FROM pos, tot
""").df()
display(balance)

,positives,rows_labelable,positive_pct
0,60436,51928255,0.1164


What I learned: Only about 0.12% of labelable drive-days are positive (60,436 of 51.9M, roughly 1 in 860), so plain accuracy is useless: a model that always says “healthy” scores about 99.88%. I’ll judge models with PR-AUC and precision at top K instead.

## Q4. Missingness map
Which SMART raw attributes are populated for which models? One day (2026-06-15) and the 12 biggest models, since reporting is the same all through the day.

In [5]:
DAY = "2026-06-15"
top = con.sql(f"""
    SELECT model FROM {SRC} WHERE date = '{DAY}'
    GROUP BY model ORDER BY count(*) DESC LIMIT 12
""").df()["model"].tolist()

all_cols = con.sql(f"SELECT * FROM {SRC} LIMIT 0").columns
raw_cols = [c for c in all_cols if c.startswith("smart_") and c.endswith("_raw")]

in_list = ", ".join("'" + m.replace("'", "''") + "'" for m in top)
select_cols = ", ".join(f'count("{c}") AS "{c}"' for c in raw_cols)
miss = con.sql(f"""
    SELECT model, count(*) AS n, {select_cols}
    FROM {SRC} WHERE date = '{DAY}' AND model IN ({in_list})
    GROUP BY model
""").df().set_index("model")

share = miss[raw_cols].div(miss["n"], axis=0)
share = share.loc[:, (share > 0).any()]          # drop columns empty for every model
print(f"{share.shape[1]} of {len(raw_cols)} raw SMART columns are populated for at least one of these models")
px.imshow(share, aspect="auto", zmin=0, zmax=1,
          title="Share of drives reporting each SMART raw attribute").show()

40 of 93 raw SMART columns are populated for at least one of these models


What I learned: Of 93 raw SMART columns, only 40 are populated for at least one of the 12 biggest models, and reporting is nearly all-or-nothing per model. A core set (for example smart_5, 197 and 198) is reported by every model, while others (for example 187, 188 and 190) appear only for the Seagate models. So a missing value usually means “this model doesn’t report it”, not a random gap. I’ll add a missing flag per attribute and a model feature, and I won’t impute a mean.

## Phase 1 gate
- [ ] One year sits in Parquet (currently half a year; download 2025 Q3 and Q4 to complete it)
- [✔️ ] You can state the failure rate per model (Q2)
- [ ] Failure counts per quarter checked against Backblaze's published quarterly reports
- [✔️ ] My quarterly failure counts match Backblaze’s published reports (Q1 1,030 exact; Q2 1,522 vs 1,498, the difference being excluded boot and small-model drives).